# Validación cruzada agrupada sobre Jamaica — issue #43

[Abrir en Colab](https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/18_validacion_cruzada_jamaica.ipynb)

Cinco folds sobre las 172 imágenes de Jamaica, agrupados por zona espacial. El split oficial 738/162
no se modifica. **El test de Kingston no se carga ni interviene en ninguna decisión.**

Cada fold entrena con el train no jamaicano más los otros cuatro folds de Jamaica. Los checkpoints y
el early stopping usan **solo el val oficial no jamaicano** (139 imágenes); el fold evaluado no elige épocas.
Esto conserva los hiperparámetros de la referencia, pero selecciona checkpoints fuera de Jamaica.
Val se usó para elegir los finalistas y también para reportar: la CV no elimina todo ese sesgo.

**Revisión de datos:** el equipo decidió el 06/11/2026 que N06 no tiene paneles. Se conserva su etiqueta
original vacía y la decisión está registrada en `docs/revision_anotaciones.csv` y `docs/criterios_datos.md`.
El código sigue frenando ante cualquier etiqueta pendiente de Jamaica. N16 permanece pendiente en la
validación no jamaicana, con su referencia provisional. No hay métricas de CV hasta ejecutar las corridas reales.

Una arquitectura por sesión nueva de Colab con GPU T4: ResNet50, YOLO11n y YOLO26n; MobileNetV2 es opcional.
Son 15 a 20 entrenamientos (las CNN tienen dos etapas). YOLO11n usa Ultralytics 8.3.221 y YOLO26n 8.4.184;
no abrir pesos de YOLO26n con la versión anterior. Esa diferencia de librería limita la comparación.

In [ ]:
# Cambiar solo estas opciones. Para inspeccionar resultados no se necesita GPU.
ENTRENAR = False
MODELO = "resnet50"  # resnet50, yolo11n, yolo26n; mobilenetv2 opcional
FOLDS = [1, 2, 3, 4, 5]  # Para recuperar un corte, seleccionar solo ese fold.
REPETIR = False  # True archiva únicamente los folds seleccionados, incluidos los completos.
RAMA = "main"  # Antes del merge del PR: codigo/validacion-cruzada-jamaica

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Abrir este notebook en Google Colab; GPU T4 solo para entrenar.")
repo = Path("/content/PDI-TP")
if repo.exists():
    rama_actual = subprocess.check_output(["git", "-C", str(repo), "branch", "--show-current"], text=True).strip()
    if rama_actual != RAMA:
        raise RuntimeError("El clon usa otra rama. Empezar una sesión nueva de Colab para esta rama.")
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", RAMA, "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
if MODELO not in ("resnet50", "mobilenetv2", "yolo11n", "yolo26n"):
    raise ValueError("Elegir un finalista del issue #43.")
requisitos = "requirements.txt"
if ENTRENAR:
    requisitos = ("requirements-yolo26.txt" if MODELO == "yolo26n" else
                  "requirements-detection.txt" if MODELO == "yolo11n" else "requirements-training.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / requisitos)], check=True)

import gc
import json
import platform
import shutil
import tempfile
import time
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pdi_tp import clasificacion as clf, comparacion, cv_jamaica, deteccion, entorno, split

rutas = entorno.preparar(solo_lectura=not ENTRENAR)
if ENTRENAR and MODELO in cv_jamaica.VERSIONES_YOLO:
    import torch
    from ultralytics import YOLO
    if version("ultralytics") != cv_jamaica.VERSIONES_YOLO[MODELO]:
        raise RuntimeError("Reiniciar la sesión después de instalar la versión correcta de Ultralytics.")
    if not torch.cuda.is_available():
        raise RuntimeError("Activar GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.")
elif ENTRENAR:
    import tensorflow as tf
    from tensorflow import keras
    from pdi_tp import aumentacion
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        raise RuntimeError("Activar GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.experimental.enable_op_determinism()

## Auditar y fijar particiones

La función compartida audita las etiquetas y verifica separación espacial y pertenencia al split.
La semilla de los folds es 42. Se aplica una permutación reproducible de los IDs de grupo y
`StratifiedGroupKFold(shuffle=False)` para conservar la correspondencia entre grupos y recuentos.
La estratificación es aproximada: hay zonas grandes de una sola clase. Revisar la distribución
mostrada antes de entrenar. Se registran los nombres, grupos, roles y hashes de datos por fold.
Los pesos de muestra se recalculan únicamente con el entrenamiento de cada fold.

In [ ]:
tabla, cajas_reales = clf.preparar_tabla(rutas)
display(tabla.loc[tabla.pendiente_revision, ["archivo", "split", "y_original", "y"]])
# Frena ante nuevas etiquetas pendientes de Jamaica; N06 ya fue resuelto por el equipo.
folds = split.folds_jamaica(rutas, n=5, semilla=42)
HUELLA_DATOS = clf.huella_tabla(tabla)
display(pd.DataFrame([dict(fold=f["fold"], conjunto=c, n=len(f[c]),
    con_paneles=int(f[c].y_original.sum()), grupos=f[c].grupo.nunique())
    for f in folds for c in ("train", "validacion", "evaluacion")]))

## Entrenamiento con la receta guardada

Se exige una referencia completa y vigente en Drive. Para CNN también se consulta la receta de la cabeza
congelada. **No se reutilizan sus pesos:** cada fold empieza desde ImageNet o COCO, y el fine-tuning
parte de la cabeza entrenada dentro de ese fold. Usar pesos de las corridas originales produciría fuga.

Se guarda solo en `analisis/cv_jamaica/<modelo>/<modo>/fold_N/`. Cada fold completo incluye pesos,
`config.json`, `particiones.csv`, `entrenamiento.json`, `predicciones_oof.csv` y un sello de integridad;
los detectores también tienen `cajas_oof.csv`. Queda fuera de `10_comparacion_final`.

Las mejores CNN se restauran desde memoria, incluso al agotar las épocas, y los checkpoints de YOLO
se escriben en el disco local de Colab. El resultado final se copia a Drive una vez terminado el fold.
**Un corte pierde el progreso del fold en curso.** Se conservan los folds completos. Para recuperar:
sesión nueva, misma arquitectura y commit, seleccionar solo el fold incompleto y `REPETIR=True` para
archivarlo y entrenarlo desde el principio. Volver a `REPETIR=False` después. No borrar carpetas.

In [ ]:
def dataset_cnn(tabla_fold, receta, entrenamiento=False):
    pesos = (clf.pesos_muestra_fuente_clase(tabla_fold.fuente, tabla_fold.y_original)
             if entrenamiento else np.ones(len(tabla_fold)))
    ds = tf.data.Dataset.from_tensor_slices((tabla_fold.ruta.to_list(),
        tabla_fold.y_original.to_numpy("float32"), np.asarray(pesos, dtype="float32")))
    if entrenamiento:
        ds = ds.shuffle(len(tabla_fold), seed=receta["semilla"], reshuffle_each_iteration=True)
    def leer(ruta, etiqueta, peso):
        imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
        imagen = tf.image.resize(imagen, (receta["img_size"], receta["img_size"]))
        if entrenamiento:
            imagen = aumentacion.google_maps(imagen)
        return imagen, tf.reshape(etiqueta, (1,)), peso
    return ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(receta["batch_size"]).prefetch(tf.data.AUTOTUNE)


def construir_cnn(nombre, receta):
    lado = receta["img_size"]
    constructor, preprocesar = {
        "resnet50": (keras.applications.ResNet50, keras.applications.resnet.preprocess_input),
        "mobilenetv2": (keras.applications.MobileNetV2, keras.applications.mobilenet_v2.preprocess_input),
    }[nombre]
    base = constructor(include_top=False, weights="imagenet", input_shape=(lado, lado, 3))
    base.trainable = False
    entrada = keras.Input((lado, lado, 3), name="rgb_0_255")
    x = base(preprocesar(entrada), training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(receta["dropout"], seed=receta["semilla"] + 2)(x)
    salida = keras.layers.Dense(1, activation="sigmoid", name="con_paneles")(x)
    return keras.Model(entrada, salida), base


def entrenar_etapa_cnn(modelo, train_ds, validacion_ds, receta, etapa, local):
    class MejorEnMemoria(keras.callbacks.Callback):
        def __init__(self):
            super().__init__()
            self.mejor, self.pesos, self.epoca = float("inf"), None, None
        def on_epoch_end(self, epoch, logs=None):
            valor = (logs or {}).get("val_loss", float("nan"))
            if np.isfinite(valor) and valor < self.mejor:
                self.mejor, self.epoca = float(valor), epoch + 1
                self.pesos = [np.array(p, copy=True) for p in self.model.get_weights()]
    mejor = MejorEnMemoria()
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=receta[f"lr_{etapa}"]),
        loss=receta["loss"], metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
    historia = modelo.fit(train_ds, validation_data=validacion_ds,
        epochs=receta[f"epocas_{etapa}"], callbacks=[mejor,
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=receta["paciencia"], restore_best_weights=True),
            keras.callbacks.TerminateOnNaN()], verbose=2)
    if mejor.pesos is None or not all(np.isfinite(v).all() for v in historia.history.values()):
        raise RuntimeError("Entrenamiento no finito; no registrar el fold.")
    modelo.set_weights(mejor.pesos)
    pd.DataFrame(historia.history).to_csv(local / f"historia_{etapa}.csv", index_label="epoca_desde_0")
    return dict(epocas=len(historia.history["loss"]), mejor_epoca=mejor.epoca,
                mejor_val_loss=mejor.mejor, monitor="val_loss_no_jamaica")


def entrenar_cnn(fold, receta, local):
    keras.backend.clear_session()
    keras.utils.set_random_seed(receta["semilla"])
    train_ds = dataset_cnn(fold["train"], receta, entrenamiento=True)
    validacion_ds = dataset_cnn(fold["validacion"], receta)
    modelo, base = construir_cnn(MODELO, receta)
    etapas = {}
    etapas["cabeza"] = entrenar_etapa_cnn(modelo, train_ds, validacion_ds, receta, "cabeza", local)
    base.trainable = True
    for capa in base.layers:
        capa.trainable = not isinstance(capa, keras.layers.BatchNormalization)
    etapas["fine_tuning"] = entrenar_etapa_cnn(modelo, train_ds, validacion_ds, receta, "fine_tuning", local)
    prob = modelo.predict(dataset_cnn(fold["evaluacion"], receta), verbose=0).reshape(-1)
    modelo.save(local / "modelo.keras")
    return prob, None, dict(etapas=etapas)

In [ ]:
def entrenar_detector(fold, receta, local, derivado):
    # Las rutas apuntan a la copia derivada, respetando el split original de cada imagen.
    def ruta_imagen(fila):
        return (derivado / "images" / fila.split / fila.archivo).as_posix()
    for conjunto in ("train", "validacion"):
        imagenes = [ruta_imagen(f) for f in fold[conjunto].itertuples(index=False)]
        (local / f"{conjunto}.txt").write_text("\n".join(imagenes) + "\n", encoding="utf-8")
    datos_yaml = local / "datos.yaml"
    datos_yaml.write_text(json.dumps(dict(path=local.as_posix(), train="train.txt", val="validacion.txt",
        names={0: "panel"})), encoding="utf-8")
    estado_mejor = {}
    def registrar_mejor(entrenador):
        # Mismo criterio y desempate que save_model en ambas versiones fijadas de Ultralytics.
        if entrenador.fitness == entrenador.best_fitness:
            estado_mejor.update(mejor_epoca=int(entrenador.epoch) + 1, fitness=float(entrenador.fitness))
    modelo = YOLO(f"{MODELO}.pt")  # Siempre COCO; nunca pesos entrenados con Jamaica.
    modelo.add_callback("on_model_save", registrar_mejor)
    modelo.train(data=str(datos_yaml), project=str(local), name="entrenamiento",
                 exist_ok=False, freeze=0, **receta)
    carpeta_entrenamiento = local / "entrenamiento"
    historia = pd.read_csv(carpeta_entrenamiento / "results.csv")
    if not estado_mejor or not np.isfinite(historia.select_dtypes(include="number").to_numpy()).all():
        raise RuntimeError("Faltan mejores épocas o el entrenamiento no es finito.")
    shutil.copy2(carpeta_entrenamiento / "weights" / "best.pt", local / "modelo.pt")
    shutil.copy2(carpeta_entrenamiento / "results.csv", local / "historia_deteccion.csv")
    shutil.copy2(carpeta_entrenamiento / "args.yaml", local / "args.yaml")
    modelo = YOLO(str(local / "modelo.pt"))
    fuentes = [ruta_imagen(f) for f in fold["evaluacion"].itertuples(index=False)]
    cabeza = {"nms": None} if MODELO == "yolo26n" else {}
    salidas = modelo.predict(source=fuentes, stream=True, imgsz=receta["imgsz"], batch=receta["batch"],
        device=0, conf=receta["conf"], iou=receta["iou"], max_det=receta["max_det"],
        augment=False, half=False, save=False, verbose=False, **cabeza)
    prob, filas = deteccion.resumir_predicciones(salidas, fold["evaluacion"].archivo)
    return prob, filas, dict(etapas={"deteccion": dict(epocas=len(historia),
        mejor_epoca=estado_mejor["mejor_epoca"], fitness=estado_mejor["fitness"], monitor="fitness_no_jamaica")})

In [ ]:
if ENTRENAR:
    if not FOLDS or len(FOLDS) != len(set(FOLDS)) or any(type(f) is not int or f not in range(1, 6) for f in FOLDS):
        raise ValueError("FOLDS debe seleccionar números únicos del 1 al 5.")
    ref = cv_jamaica.referencia(rutas, MODELO)
    paquetes = (["ultralytics", "torch", "torchvision", "numpy", "pandas", "Pillow", "scikit-learn"]
                if MODELO in cv_jamaica.VERSIONES_YOLO else
                ["tensorflow", "keras", "numpy", "pandas", "Pillow", "scikit-learn"])
    gpu = torch.cuda.get_device_name(0) if MODELO in cv_jamaica.VERSIONES_YOLO else str(gpus)
    params = dict(ref, huella_datos=HUELLA_DATOS, semilla_folds=42,
        estrategia_folds=split.ESTRATEGIA_FOLDS_JAMAICA,
        entorno=dict(python=platform.python_version(), gpu=gpu, paquetes={p: version(p) for p in paquetes}))
    modo = cv_jamaica.FINALISTAS[MODELO]
    derivado = repo / "data" / "yolo_paneles_cv_v1"
    if MODELO in cv_jamaica.VERSIONES_YOLO:
        deteccion.preparar_dataset(rutas, derivado)
    for fold in folds:
        if fold["fold"] not in FOLDS:
            continue
        if not REPETIR and cv_jamaica.fold_completo(rutas, MODELO, modo, fold, params):
            print("Se reutiliza el fold íntegro:", MODELO, fold["fold"])
            continue
        carpeta = cv_jamaica.preparar_fold(rutas, MODELO, modo, fold, params, reiniciar=REPETIR)
        local = Path(tempfile.mkdtemp(prefix=f"cv_{MODELO}_fold_{fold['fold']}_", dir="/content"))
        inicio = time.perf_counter()
        if MODELO in cv_jamaica.VERSIONES_YOLO:
            prob, filas, entrenamiento = entrenar_detector(fold, ref["receta"], local, derivado)
        else:
            prob, filas, entrenamiento = entrenar_cnn(fold, ref["receta"], local)
        entrenamiento["segundos"] = time.perf_counter() - inicio
        for archivo in local.iterdir():
            if archivo.is_file() and (archivo.suffix in (".keras", ".pt", ".csv") or archivo.name == "args.yaml"):
                shutil.copy2(archivo, carpeta / archivo.name)
        cv_jamaica.guardar_fold(rutas, carpeta, MODELO, modo, fold, params, prob, entrenamiento, cajas=filas)
        del prob, filas
        gc.collect()
        if MODELO in cv_jamaica.VERSIONES_YOLO:
            torch.cuda.empty_cache()
else:
    print("Modo análisis: Drive abierto en solo lectura; no se entrena ni se guardan archivos.")

## Comparación sobre predicciones fuera de fold

**Regla fijada antes de mirar resultados:** por familia, comparar balanced accuracy a los umbrales
actuales (0.5 CNN; 0.25 YOLO). La diferencia debe tener un intervalo pareado que no incluya cero.
Si no es clara, preferir el modelo más simple o de menor costo medido y explicarlo en el informe.
No declarar ganador si faltan candidatos o si no se midió su costo.

Cambiar el umbral solo si mejora claramente la estimación por grupos (`*_cv`) frente al actual.
El umbral propuesto con todas las imágenes no se evalúa sobre esas mismas imágenes como evidencia
de mejora. La validación de umbrales deja grupos afuera, pero los modelos de los otros folds comparten
datos: **no es una CV anidada de todo el proceso**. Los intervalos remuestrean imágenes y no incluyen
la variación entre entrenamientos; imágenes vecinas también pueden hacerlos demasiado estrechos.
La decisión se documenta en `docs/evaluacion_jamaica.md` antes de volver a evaluar el test una vez.

Val eligió los finalistas y se usó para reportar sus resultados originales. El val no jamaicano
elige ahora checkpoints, y Jamaica fuera de fold se usa para comparar; no es el test independiente.

In [ ]:
predicciones, estado = cv_jamaica.cargar_predicciones(rutas, folds)
display(estado)
if predicciones.empty:
    print("Todavía no hay modelos con sus cinco folds completos; no hay métricas para comparar.")
else:
    corridas = (predicciones.modelo + "/" + predicciones.modo).unique().tolist()
    resumen = comparacion.intervalos(predicciones, corridas, metricas=(*comparacion.METRICAS, "recall"))
    resumen["familia"] = resumen.corrida.str.split("/").str[0].map(comparacion.familia)
    display(resumen.sort_values(["familia", "balanced_accuracy"], ascending=[True, False]))
    for familia, candidatos in resumen.groupby("familia"):
        orden = candidatos.sort_values("balanced_accuracy", ascending=False).corrida.tolist()
        if len(orden) < 2:
            print("Falta otro candidato completo para la diferencia pareada:", familia)
        else:
            display(comparacion.diferencias(predicciones, orden[:2], referencia=orden[1], metrica="balanced_accuracy"))
    display(cv_jamaica.tabla_umbrales(predicciones))
    costos = []
    for modelo, modo in predicciones[["modelo", "modo"]].drop_duplicates().itertuples(index=False):
        for f in folds:
            archivo = cv_jamaica.ruta_fold(rutas, modelo, modo, f["fold"]) / "entrenamiento.json"
            registro = json.loads(archivo.read_text(encoding="utf-8"))
            costos.append(dict(modelo=modelo, modo=modo, fold=f["fold"], segundos=registro["segundos"],
                mejores_epocas={etapa: datos["mejor_epoca"] for etapa, datos in registro["etapas"].items()}))
    print("Costo por fold (incluye entrenamiento e inferencia); revisar el hardware y entorno en config.json:")
    display(pd.DataFrame(costos))
    fig, ax = plt.subplots(figsize=(8, max(2, len(resumen) * 0.65)))
    for i, fila in resumen.iterrows():
        ax.errorbar(fila.balanced_accuracy, i, xerr=[[max(0, fila.balanced_accuracy - fila.balanced_accuracy_inf)],
            [max(0, fila.balanced_accuracy_sup - fila.balanced_accuracy)]], fmt="o")
    ax.set(yticks=range(len(resumen)), yticklabels=resumen.corrida, xlim=(0, 1),
           xlabel="Balanced accuracy; intervalo bootstrap 95 % (imágenes)", title="Jamaica fuera de fold")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# mAP agrupado sobre las cajas fuera de fold de Jamaica; no promediar los mAP de los folds.
filas_deteccion = []
for modelo in cv_jamaica.VERSIONES_YOLO:
    if modelo not in set(predicciones.modelo):
        continue
    modo = cv_jamaica.FINALISTAS[modelo]
    pred = predicciones.loc[predicciones.modelo.eq(modelo)]
    reales = {archivo: cajas_reales.loc[cajas_reales.incluida & cajas_reales.archivo.eq(archivo),
        ["xc", "yc", "w", "h"]].to_numpy(float) for archivo in pred.archivo}
    cajas_oof = pd.concat([pd.read_csv(cv_jamaica.ruta_fold(rutas, modelo, modo, f["fold"]) / "cajas_oof.csv")
                          for f in folds], ignore_index=True)
    filas_deteccion.append(dict(corrida=f"{modelo}/{modo}", n=len(pred),
        **deteccion.map_cajas(reales, cajas_oof.to_dict("records"))))
display(pd.DataFrame(filas_deteccion))

## Registrar la decisión después de correr

No hay una decisión de modelo ni de umbral precargada. Anotar en `docs/evaluacion_jamaica.md`:
fecha y commits, revisión de N06, configuraciones completas, métricas e intervalos, diferencias pareadas,
umbral actual/propuesto y estimación `*_cv`, costo medido y decisión por familia. Si faltan corridas o
las diferencias son inciertas, decirlo. Las mejores épocas quedan en cada `entrenamiento.json` para #45.

No ejecutar el test desde este notebook. `17_evaluacion_test` se corre una sola vez después de fijar
modelos y umbrales. No elegirlos mirando resultados anteriores del test.

Referencias del criterio de checkpoint: `save_model` en
[Ultralytics 8.3.221](https://github.com/ultralytics/ultralytics/blob/v8.3.221/ultralytics/engine/trainer.py)
y [8.4.184](https://github.com/ultralytics/ultralytics/blob/v8.4.184/ultralytics/engine/trainer.py).